# Step 4 – Indicator calculation
This notebook calculates the indicators of step 4 for every sensor and every 3-minute interval:

1. Pedestrian count per three-minute interval: $N_{total} = N_A + N_B$
2. Pedestrian flow per metre of effective walking width: $q = N_{total} / (3 \cdot w)$
3. Directional share: % of the flow per direction
4. Change between consecutive measurements: $\Delta N = N_t - N_{t-1}$
5. Average flows per hour, day and part of day (separate tables)
6. Base table with the indicators of 1–4

All 3-minute indicators end up in **one base table** (one row = sensor × timestamp). That table is the input for
step 5 (peaks and sudden changes) and step 6 (comparison). Peaks are **not** calculated here.

**Input:**
- `../Data/SAIL2025_LVMA_cleaned.csv` – output of `SAIL2025_cleaned.ipynb`
- `../Data/sensor-location.xlsx` – effective walking widths

**Output** in `../Data/indicators/`:
- `step4_base_indicators.csv` – base table (sensor × 3 min)
- `step4_average_flow_per_hour.csv`, `step4_average_flow_per_day.csv`, `step4_average_flow_per_part_of_day.csv`

Run the notebook from top to bottom.

## 0. Preparation

In [ ]:
import os
import pandas as pd

DATA_DIR = os.path.join('..', 'Data')
OUT_DIR = os.path.join(DATA_DIR, 'indicators')
os.makedirs(OUT_DIR, exist_ok=True)

INTERVAL_MIN = 3     # one measurement interval = 3 minutes

### 0.1 Load the cleaned count data
After reading the CSV the timestamp has to be converted to datetime again.

In [ ]:
counts = pd.read_csv(os.path.join(DATA_DIR, 'SAIL2025_LVMA_cleaned.csv'))
counts['timestamp'] = pd.to_datetime(counts['timestamp'])

time_cols = ['hour', 'minute', 'day', 'month', 'weekday', 'is_weekend']
sensor_cols = [c for c in counts.columns if c not in time_cols and c != 'timestamp']

print("Shape:", counts.shape)
print("Sensor columns:", len(sensor_cols))

### 0.2 Load the sensor locations and effective widths
Same cleaning as in `sensor-location_cleaned.ipynb`. The widths are stored as text with a decimal
comma (`"3,1"`) and are converted to numbers, otherwise you cannot divide by them.

In [ ]:
loc = pd.read_excel(os.path.join(DATA_DIR, 'sensor-location.xlsx'))

# column names: replace \xa0 and double spaces by one normal space
loc.columns = loc.columns.str.replace(r'\s+', ' ', regex=True).str.strip()
loc = loc.rename(columns={'Objectummer': 'sensor',
                          'Locatienaam': 'location_name',
                          'Effectieve breedte': 'eff_width_m'})
loc['sensor'] = loc['sensor'].astype(str).str.strip()

# GASA-06-B gets the location of GASA-06 (Azartplein B); GVCV-09 is not in the cleaned counts
loc.loc[loc['sensor'] == 'GASA-06', 'sensor'] = 'GASA-06-B'
loc = loc[loc['sensor'] != 'GVCV-09']

# decimal comma -> number
loc['eff_width_m'] = pd.to_numeric(loc['eff_width_m'].astype(str).str.replace(',', '.'), errors='coerce')

loc = loc[['sensor', 'location_name', 'eff_width_m']].reset_index(drop=True)
loc

### 0.3 From wide to long format
In the count data every sensor has two columns `SENSORID_DIRECTION` (direction in degrees).
We make **one row per sensor per timestamp**, with the two directions next to each other:
- **direction A** = column with the smallest bearing (e.g. `_120`)
- **direction B** = the opposite direction (e.g. `_300`)

In [ ]:
directions = {}
for col in sensor_cols:
    sensor, deg = col.rsplit('_', 1)
    directions.setdefault(sensor, []).append(int(deg))

parts = []
for sensor, degs in directions.items():
    a, b = sorted(degs)
    parts.append(pd.DataFrame({
        'timestamp': counts['timestamp'],
        'sensor': sensor,
        'dir_A_deg': a,
        'dir_B_deg': b,
        'N_A': counts[f'{sensor}_{a}'],
        'N_B': counts[f'{sensor}_{b}'],
    }))

long = pd.concat(parts, ignore_index=True)
long = long.merge(loc, on='sensor', how='left')
long = long.sort_values(['sensor', 'timestamp']).reset_index(drop=True)

print("Rows:", len(long), "=", long['sensor'].nunique(), "sensors x", long['timestamp'].nunique(), "timestamps")
print("Sensors not in location file:", sorted(set(long['sensor']) - set(loc['sensor'])))
print("Sensors without effective width:", long.loc[long['eff_width_m'].isna(), 'sensor'].unique())
long.head()

## 1. Pedestrian count per three-minute interval

$$N_{total} = N_A + N_B$$

Both directions measure the same passage, so they are added up. The total is **only** calculated if **both**
directions are available; otherwise it stays NaN (e.g. CMSA-GAWW-20 after its failure on 23 August).

In [ ]:
# min_count=2: if one direction is NaN, the total is also NaN
long['N_total'] = long[['N_A', 'N_B']].sum(axis=1, min_count=2)

print("Missing totals per sensor:")
print(long.loc[long['N_total'].isna(), 'sensor'].value_counts())

## 2. Pedestrian flow per metre of effective walking width

$$Q = \frac{N_{total}}{3} \quad \text{(ped/min)} \qquad\qquad q = \frac{N_{total}}{3 \cdot w} \quad \text{(ped/min/m)}$$

$w$ = effective walking width in metres. The specific flow $q$ makes wide and narrow locations comparable.
Sensors without an effective width get no $q$ (NaN).

Example from the plan: 180 pedestrians in 3 minutes (120 in A, 60 in B) at a width of 4 m → $Q$ = 60 ped/min,
$q$ = 15 ped/min/m, $Q_A$ = 40 ped/min and $Q_B$ = 20 ped/min.

In [ ]:
long['Q_total'] = long['N_total'] / INTERVAL_MIN
long['Q_A'] = long['N_A'] / INTERVAL_MIN
long['Q_B'] = long['N_B'] / INTERVAL_MIN
long['q_specific'] = long['N_total'] / (INTERVAL_MIN * long['eff_width_m'])

# check with the example from the plan
print("Example: Q =", 180 / INTERVAL_MIN, "ped/min,  q =", 180 / (INTERVAL_MIN * 4), "ped/min/m,",
      " Q_A =", 120 / INTERVAL_MIN, "ped/min,  Q_B =", 60 / INTERVAL_MIN, "ped/min")

long[['sensor', 'eff_width_m', 'N_total', 'Q_total', 'q_specific']].describe()

## 3. Directional share

$$\text{share}_A = \frac{N_A}{N_{total}} \cdot 100\% \qquad \text{share}_B = 100\% - \text{share}_A$$

Shows which direction dominates and how that shifts during the day (e.g. arrivals in the morning, departures
in the evening).

Example from the plan: 120 of 180 pedestrians in direction A → A = 67%, B = 33%. If $N_{total} = 0$ the share is undefined and stays NaN.

In [ ]:
# check with the example from the plan
print("Example: A =", round(120 / 180 * 100), "%,  B =", round(60 / 180 * 100), "%")

valid_total = long['N_total'].where(long['N_total'] > 0)
long['share_A_pct'] = long['N_A'] / valid_total * 100
long['share_B_pct'] = long['N_B'] / valid_total * 100

# average share per sensor over the busy intervals only (quiet night hours say little)
busy = long['N_total'] >= 30
print("Average share of direction A when busy (>= 30 ped / 3 min):")
print(long[busy].groupby('sensor')['share_A_pct'].mean().round(1).sort_values().to_string())

## 4. Changes in pedestrian flow between consecutive measurements

$$\Delta N = N_t - N_{t-1}$$

Calculated for the total and for each direction. Positive = increase, negative = decrease.
A change is **only** calculated between two **valid** measurements of the **same sensor** that are
**exactly 3 minutes apart**. Otherwise $\Delta N$ stays NaN (first measurement of a sensor, missing value, gap in time).

The relative change `dN_total_pct` is also stored; it is NaN if the previous count was 0.

Deciding *which* changes are "sudden" is step 5.

In [ ]:
per_sensor = long.groupby('sensor')

# True if the previous row of the same sensor is exactly 3 minutes earlier
exactly_3min = per_sensor['timestamp'].diff() == pd.Timedelta(minutes=INTERVAL_MIN)

# diff() is already NaN if the current or previous value is NaN
long['dN_total'] = per_sensor['N_total'].diff().where(exactly_3min)
long['dN_A'] = per_sensor['N_A'].diff().where(exactly_3min)
long['dN_B'] = per_sensor['N_B'].diff().where(exactly_3min)

previous = per_sensor['N_total'].shift()
long['dN_total_pct'] = long['dN_total'] / previous.where(previous > 0) * 100

print("Valid changes:", long['dN_total'].notna().sum(), "of", len(long))
long[['dN_total', 'dN_A', 'dN_B']].describe()

## 5. Average flows per time period

$$Q_{average} = \frac{Q_1 + Q_2 + \dots + Q_n}{n}$$

The 3-minute flows are averaged per sensor for three time periods:
- **hour** (per date and hour)
- **day** (per date)
- **part of day**: night (00–06), morning (06–12), afternoon (12–18), evening (18–24)

Example: three measurements of 40, 50 and 60 ped/min → $Q_{average}$ = 50 ped/min.

These averages are used later to compare general patterns between locations and times. They are stored in
**separate tables**; the original 3-minute values stay in the base table for peaks and sudden changes.
`n_intervals` shows how many valid 3-minute values each average is based on (max. 20 per hour).

In [ ]:
# check with the example from the plan
print("Example: Q_average =", sum([40, 50, 60]) / 3, "ped/min")

long['date'] = long['timestamp'].dt.date
long['hour'] = long['timestamp'].dt.hour
long['part_of_day'] = pd.cut(long['hour'], bins=[0, 6, 12, 18, 24], right=False,
                             labels=['night', 'morning', 'afternoon', 'evening'])

avg_cols = ['Q_total', 'Q_A', 'Q_B', 'q_specific']

def average_flows(group_cols):
    g = long.groupby(['sensor', 'location_name'] + group_cols, observed=True)
    out = g[avg_cols].mean().add_suffix('_avg')
    out['n_intervals'] = g['Q_total'].count()
    return out.reset_index()

avg_hour = average_flows(['date', 'hour'])
avg_day = average_flows(['date'])
avg_part_of_day = average_flows(['date', 'part_of_day'])

avg_hour.to_csv(os.path.join(OUT_DIR, 'step4_average_flow_per_hour.csv'), index=False)
avg_day.to_csv(os.path.join(OUT_DIR, 'step4_average_flow_per_day.csv'), index=False)
avg_part_of_day.to_csv(os.path.join(OUT_DIR, 'step4_average_flow_per_part_of_day.csv'), index=False)

print("Rows per table: hour", len(avg_hour), "| day", len(avg_day), "| part of day", len(avg_part_of_day))
avg_day.head()

## 6. Base table
The results of sections 1–4 together in one file: one row = sensor × timestamp. This is the input for
peak and change detection (step 5) and the comparison across location and time (step 6).

In [ ]:
base_cols = ['timestamp', 'sensor', 'location_name', 'eff_width_m', 'dir_A_deg', 'dir_B_deg',
             'N_A', 'N_B', 'N_total',
             'Q_A', 'Q_B', 'Q_total', 'q_specific',
             'share_A_pct', 'share_B_pct',
             'dN_A', 'dN_B', 'dN_total', 'dN_total_pct']
base = long[base_cols]

base.to_csv(os.path.join(OUT_DIR, 'step4_base_indicators.csv'), index=False)
print("Saved:", base.shape)
base.head()